# CrisisInbox GRPO Training with Unsloth

Train a small LLM to triage crisis inbox messages using **Unsloth + GRPO**.

Connects to the live CrisisInbox environment on HuggingFace Spaces to collect training episodes, then trains with Group Relative Policy Optimization.

**Stack:** Unsloth (2x faster LoRA) + HF TRL GRPO

**What this does:**
1. Connects to deployed CrisisInbox env via WebSocket
2. Collects episodes by interacting with the environment
3. Trains with GRPO using urgency/deadline/drift reward signals (2x faster with Unsloth)
4. Evaluates before vs after training against the live env

Open in Google Colab with a **T4 GPU** runtime.

In [ ]:
# Install Unsloth (handles transformers, trl, peft, etc.) and vLLM for fast inference
!pip install unsloth vllm -q
!pip install "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv.git" -q
!pip install huggingface_hub matplotlib -q
print("Setup complete")

In [ ]:
# Patch transformers logging crash
import logging
import warnings

def _patch_transformers_logging():
    try:
        import transformers.utils.logging as trans_log
        _orig = trans_log.logger.warning
        def _safe_warning(msg, *args, **kwargs):
            if args and isinstance(args[0], type) and issubclass(args[0], Warning):
                args = ()
            return _orig(msg, *args, **kwargs)
        trans_log.logger.warning = _safe_warning
    except Exception:
        pass
    warnings.filterwarnings("ignore", message=".*attention mask API.*", category=FutureWarning)
    warnings.filterwarnings("ignore", message="remove second argument of ws_handler", category=DeprecationWarning)

_patch_transformers_logging()

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_bytes = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    vram_gb = total_bytes / 1e9 if total_bytes else 0
    if vram_gb == 0 and hasattr(torch.cuda, "mem_get_info"):
        _, total_bytes = torch.cuda.mem_get_info(0)
        vram_gb = total_bytes / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)")
else:
    print("No GPU available.")

## Connect to CrisisInbox Environment

Connect to the live environment on HuggingFace Spaces via WebSocket and collect episodes by running through the 48-hour simulation multiple times with different seeds.

In [ ]:
import json
import time as _time
from openenv.core.mcp_client import MCPToolClient

BASE_URL = "https://eptan-crisis-inbox.hf.space"

# Wake up the HF Space (may be sleeping) and verify connectivity
print("Connecting to HF Space (may take a moment if cold-starting)...")
for attempt in range(3):
    try:
        with MCPToolClient(base_url=BASE_URL, connect_timeout_s=60.0).sync() as env:
            env.reset(seed=0)
            tools = env.list_tools()
            print(f"Connected! Available tools: {[t.name for t in tools]}")
            for t in tools:
                print(f"  - {t.name}: {t.description[:80]}...")

            status = json.loads(env.call_tool("get_status"))
            print(f"\nEnvironment ready — {status['messages_total_arrived']} messages at hour {status['current_hour']}")
        break
    except Exception as e:
        if attempt < 2:
            print(f"  Attempt {attempt + 1} failed ({e}), retrying in 10s...")
            _time.sleep(10)
        else:
            raise RuntimeError(f"Could not connect to {BASE_URL} after 3 attempts: {e}")

In [ ]:
import random


def collect_episode(base_url, seed, time_steps=None):
    """Collect one episode from the live environment using OpenEnv tools."""
    if time_steps is None:
        time_steps = [0, 2, 6, 12, 18, 24, 30, 36, 42, 47]

    superseded_msgs = {}

    with MCPToolClient(
        base_url=base_url, connect_timeout_s=60.0, message_timeout_s=120.0,
    ).sync() as env:
        env.reset(seed=seed)
        decision_points = []
        current_hour = 0.0

        for target_hour in time_steps:
            while current_hour < target_hour - 0.1:
                jump = min(4.0, target_hour - current_hour)
                env.call_tool("advance_time", hours=jump)
                status = json.loads(env.call_tool("get_status"))
                current_hour = status["current_hour"]

            inbox = json.loads(env.call_tool("get_inbox"))
            prompt = env.call_tool("get_prompt")  # Server builds the prompt

            for m in inbox:
                if m.get("superseded"):
                    superseded_msgs[m["id"]] = ""

            unhandled = [m for m in inbox if not m.get("handled", False)]
            if not unhandled:
                continue

            decision_points.append({
                "hour": target_hour,
                "visible_count": len(inbox),
                "prompt": prompt,
                "messages": inbox,
                "superseded": dict(superseded_msgs),
            })

    return {
        "episode_id": f"ep_{seed}",
        "seed": seed,
        "drift_events": [],
        "superseded_messages": superseded_msgs,
        "decision_points": decision_points,
    }


# Test: collect one episode
print("Collecting test episode (seed=42)...")
test_ep = collect_episode(BASE_URL, seed=42)
print(f"Episode {test_ep['episode_id']}: {len(test_ep['decision_points'])} decision points")
for dp in test_ep["decision_points"]:
    print(f"  Hour {dp['hour']:5.1f}: {dp['visible_count']} messages visible")

In [ ]:
# Collect multiple episodes from the live environment
NUM_EPISODES = 10
SEEDS = list(range(NUM_EPISODES))

episodes = []
for seed in SEEDS:
    print(f"Collecting episode {seed + 1}/{NUM_EPISODES} (seed={seed})...", end=" ")
    for attempt in range(3):
        try:
            ep = collect_episode(BASE_URL, seed=seed)
            episodes.append(ep)
            print(f"{len(ep['decision_points'])} decision points")
            break
        except Exception as e:
            if attempt < 2:
                print(f"retry {attempt + 1}...", end=" ")
                _time.sleep(5)
            else:
                print(f"FAILED ({e}), skipping")

# Flatten to training prompts
prompts = []
for ep in episodes:
    for dp in ep["decision_points"]:
        prompts.append({
            "prompt": dp["prompt"],
            "hour": dp["hour"],
            "visible_count": dp["visible_count"],
            "episode_id": ep["episode_id"],
            "seed": ep["seed"],
            "drift_events": ep["drift_events"],
            "superseded": ep.get("superseded_messages", {}),
            "messages": dp["messages"],
        })

print(f"\nCollected {len(episodes)} episodes -> {len(prompts)} training prompts")
print(f"Average {len(prompts)/len(episodes):.1f} decision points per episode")

## Reward Function

Scores agent actions based on:
- **Urgency base** (critical=10, high=5, medium=3, low=1)
- **Deadline timing** (early=bonus, late=penalty)
- **Drift adaptation** (+50% for handling policy-change messages)
- **Stale info penalty** (-50% for acting on superseded messages)
- **Response quality** (penalty for short/empty responses)

In [ ]:
import re


def score_action(completion, prompt_data):
    """Score a model completion. Mirrors server's calculate_reward()."""
    messages = prompt_data["messages"]
    hour = prompt_data["hour"]
    superseded = prompt_data.get("superseded", {})

    match = re.search(
        r'respond_to_message\s*\(\s*["\']?(msg_\d+)["\']?\s*,\s*["\'](.+?)["\']',
        completion, re.DOTALL,
    )
    if match:
        msg_id, response_text = match.group(1), match.group(2)
    else:
        id_match = re.search(r'(msg_\d+)', completion)
        if id_match:
            msg_id, response_text = id_match.group(1), completion[:200]
        else:
            return -1.0

    target = next((m for m in messages if m["id"] == msg_id), None)
    if target is None:
        return -0.5

    urgency_rewards = {"critical": 10.0, "high": 5.0, "medium": 3.0, "low": 1.0}
    reward = urgency_rewards.get(target["urgency"], 1.0)

    deadline = target.get("deadline_hours")
    if deadline is not None:
        if hour <= deadline:
            reward *= 1.0 + 0.5 * ((deadline - hour) / max(deadline, 1.0))
        else:
            reward *= 0.25

    if len(response_text.strip()) < 10:
        reward *= 0.5

    if target.get("drift_flag"):
        reward *= 1.5

    if target["id"] in superseded:
        reward *= 0.5

    unhandled = [m for m in messages if not m.get("handled") and m["id"] != msg_id]
    if any(m["urgency"] == "critical" for m in unhandled) and target["urgency"] in ("low", "medium"):
        reward *= 0.3

    return round(reward, 2)


# Test
test_data = prompts[0]
print("Testing reward function:")
print(f"  Hour: {test_data['hour']}, Messages: {test_data['visible_count']}")
critical_msgs = [m for m in test_data["messages"] if m["urgency"] == "critical"]
if critical_msgs:
    good = f'respond_to_message("{critical_msgs[0]["id"]}", "Evacuating now with documents.")'
    print(f"  Good action (critical): {score_action(good, test_data):.2f}")
low_msgs = [m for m in test_data["messages"] if m["urgency"] == "low"]
if low_msgs:
    bad = f'respond_to_message("{low_msgs[0]["id"]}", "ok")'
    print(f"  Bad action (low, short): {score_action(bad, test_data):.2f}")
print(f"  Unparseable: {score_action('do something', test_data):.2f}")

## Load Model with Unsloth & Baseline Evaluation

Load Qwen2.5-0.5B using Unsloth's `FastLanguageModel` for 2x faster LoRA training. Unsloth handles 4-bit quantization, LoRA patching, and TRL integration automatically.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024

# Unsloth handles quantization, dtype selection, and device mapping automatically
# fast_inference requires vLLM which may not be available — skip it
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

# Apply LoRA adapters via Unsloth (patches for 2x speed)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
)

# GRPO requires left padding so completions align across the batch
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded with Unsloth (4-bit quantized)")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")

### Pre-Training Baseline

Evaluate the **untrained** model against the live environment before any GRPO training. This gives us a baseline to measure improvement.

In [ ]:
# --- Pre-training baseline evaluation against live environment ---
from openenv.core.env_server.mcp_types import CallToolAction


def generate_action(model, tokenizer, prompt_text):
    """Generate an action from the model."""
    # Ensure model is in inference mode for fast generation
    FastLanguageModel.for_inference(model)
    msgs = [{"role": "user", "content": prompt_text}]
    input_ids = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True)
    if not isinstance(input_ids, torch.Tensor):
        input_ids = input_ids["input_ids"]
    input_ids = input_ids.to("cuda")
    prompt_len = input_ids.shape[1]
    with torch.no_grad():
        output = model.generate(input_ids=input_ids, max_new_tokens=200, temperature=0.7,
                                pad_token_id=tokenizer.pad_token_id, do_sample=True)
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)


def _extract_tool_result(obs):
    """Extract JSON from a CallToolObservation (handles FastMCP wrapping)."""
    raw = getattr(obs, "result", None)
    if hasattr(raw, "data"):
        raw = raw.data
    if isinstance(raw, dict) and "data" in raw:
        raw = raw["data"]
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except (json.JSONDecodeError, TypeError):
            return {}
    return raw if isinstance(raw, dict) else {}


def evaluate_on_live_env(model, tokenizer, base_url, seed, max_steps=20):
    """Evaluate model against the live environment using OpenEnv step() flow."""
    with MCPToolClient(base_url=base_url, connect_timeout_s=60.0, message_timeout_s=120.0).sync() as env:
        env.reset(seed=seed)
        total_reward = 0.0
        actions_taken = []

        for step_i in range(max_steps):
            status = json.loads(env.call_tool("get_status"))
            if status.get("done"):
                break

            inbox = json.loads(env.call_tool("get_inbox"))
            current_hour = status["current_hour"]
            unhandled = [m for m in inbox if not m.get("handled", False)]
            if not unhandled:
                env.call_tool("advance_time", hours=2.0)
                continue

            # Use server's get_prompt tool
            prompt = env.call_tool("get_prompt")
            completion = generate_action(model, tokenizer, prompt)

            match = re.search(r'respond_to_message\s*\(\s*["\']?(msg_\d+)["\']?\s*,\s*["\'](.+?)["\']', completion, re.DOTALL)
            if not match:
                id_match = re.search(r'(msg_\d+)', completion)
                if id_match:
                    msg_id, response_text = id_match.group(1), completion[:200]
                else:
                    env.call_tool("advance_time", hours=1.0)
                    continue
            else:
                msg_id, response_text = match.group(1), match.group(2)

            action = CallToolAction(
                tool_name="respond_to_message",
                arguments={"message_id": msg_id, "response": response_text},
            )
            step_result = env.step(action)
            obs = step_result.observation
            reward = obs.reward if obs.reward is not None else 0.0
            done = obs.done

            result_data = _extract_tool_result(obs)
            if "error" in result_data:
                env.call_tool("advance_time", hours=1.0)
                continue

            total_reward += reward
            target_msg = next((m for m in inbox if m["id"] == msg_id), None)
            urgency = target_msg["urgency"] if target_msg else "?"
            actions_taken.append({"step": step_i, "hour": current_hour, "msg_id": msg_id, "urgency": urgency, "reward": reward})
            print(f"  Step {step_i:2d} | Hour {current_hour:5.1f} | {msg_id} ({urgency:8s}) | Reward: {reward:+.1f} | Total: {total_reward:.1f}")

            if done:
                break

        final_status = json.loads(env.call_tool("get_status"))

    return {"seed": seed, "total_reward": total_reward, "actions": actions_taken, "final_status": final_status}


# Run baseline on 3 seeds
print("=== PRE-TRAINING BASELINE (untrained model) ===\n")
baseline_results = []
for seed in [99, 42, 7]:
    print(f"--- Seed {seed} ---")
    res = evaluate_on_live_env(model, tokenizer, BASE_URL, seed=seed)
    baseline_results.append(res)
    print(f"  Total: {res['total_reward']:.1f} | Actions: {len(res['actions'])}\n")

baseline_avg = sum(r["total_reward"] for r in baseline_results) / len(baseline_results)
print(f"Baseline average reward: {baseline_avg:.1f}")

In [ ]:
from datasets import Dataset

MAX_PROMPT_LENGTH = 1024

train_data = []
for p in prompts:
    msgs = [{"role": "user", "content": p["prompt"]}]
    tok = tokenizer.apply_chat_template(msgs, truncation=True, max_length=1024, return_tensors="pt", add_generation_prompt=True)
    try:
        ids = tok["input_ids"]
    except (TypeError, KeyError):
        ids = tok
    n_tokens = ids.shape[1] if ids.dim() > 1 else ids.shape[0]
    if n_tokens > MAX_PROMPT_LENGTH:
        continue
    train_data.append({
        "prompt": msgs,
        "_prompt_key": p["prompt"][:200],
    })

random.seed(42)
random.shuffle(train_data)

dataset = Dataset.from_list(train_data)
print(f"Training dataset: {len(dataset)} prompts (after dropping prompts > {MAX_PROMPT_LENGTH} tokens)")
print(f"Sample prompt length: {len(train_data[0]['prompt'][0]['content'])} chars")

## GRPO Training Loop

Unsloth patches TRL's `GRPOTrainer` for ~2x faster training with automatic gradient checkpointing and fused kernels. No need to pass `peft_config` separately — LoRA is already applied.

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Build lookup from prompt text -> prompt metadata for reward scoring
prompt_lookup = {}
for p in prompts:
    key = p["prompt"][:200]
    prompt_lookup[key] = p


def reward_fn(prompts, completions, **kwargs):
    """GRPO reward function. Scores each completion against its inbox state."""
    rewards = []
    for prompt_msgs, completion in zip(prompts, completions):
        # Extract prompt text to look up metadata
        if isinstance(prompt_msgs, list):
            prompt_text = prompt_msgs[-1]["content"] if prompt_msgs else ""
        else:
            prompt_text = str(prompt_msgs)

        key = prompt_text[:200]
        prompt_data = prompt_lookup.get(key)

        if prompt_data is None:
            rewards.append(0.0)
            continue

        if isinstance(completion, list):
            if completion and isinstance(completion[0], (int, float)):
                comp_text = tokenizer.decode(completion, skip_special_tokens=True)
            else:
                comp_text = completion[-1].get("content", "") if completion else ""
        else:
            comp_text = str(completion)

        score = score_action(comp_text, prompt_data)
        rewards.append(score)

    return rewards


training_args = GRPOConfig(
    output_dir="crisisinbox-grpo-unsloth-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    max_completion_length=256,
    max_prompt_length=max_seq_length,
    num_generations=4,
    logging_steps=1,
    save_steps=100,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    fp16=not (torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False),
)

# Unsloth: LoRA is already applied to the model, so no peft_config needed
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=dataset,
)

# After trainer init, update model ref
model = trainer.model

print(f"Trainer configured — {len(prompt_lookup)} unique prompt keys")
print(f"Training for {training_args.num_train_epochs} epochs")
print("Ready to train (Unsloth 2x speedup active)")

In [ ]:
# Train!
trainer.train()
print("Training complete")

## Evaluate: Offline + Training Curve

Evaluate the trained model on collected prompts and plot the training reward curve.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Post-training evaluation on same seeds as baseline ---
print("=== POST-TRAINING EVALUATION ===\n")
trained_results = []
for seed in [99, 42, 7]:
    print(f"--- Seed {seed} ---")
    res = evaluate_on_live_env(model, tokenizer, BASE_URL, seed=seed)
    trained_results.append(res)
    print(f"  Total: {res['total_reward']:.1f} | Actions: {len(res['actions'])}\n")

trained_avg = sum(r["total_reward"] for r in trained_results) / len(trained_results)
print(f"Trained average reward: {trained_avg:.1f}")
print(f"Baseline average reward: {baseline_avg:.1f}")
improvement = ((trained_avg - baseline_avg) / max(baseline_avg, 0.1)) * 100
print(f"Improvement: {improvement:+.1f}%")

# --- Plot 1: Before/After Comparison Bar Chart ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Bar chart: per-seed comparison
seeds = [99, 42, 7]
baseline_scores = [r["total_reward"] for r in baseline_results]
trained_scores = [r["total_reward"] for r in trained_results]

x = range(len(seeds))
width = 0.35
bars1 = axes[0].bar([i - width/2 for i in x], baseline_scores, width, label="Before Training", color="#d62728", alpha=0.8)
bars2 = axes[0].bar([i + width/2 for i in x], trained_scores, width, label="After Training", color="#2ca02c", alpha=0.8)
axes[0].set_xlabel("Episode Seed")
axes[0].set_ylabel("Total Reward")
axes[0].set_title("Before vs After GRPO Training (Unsloth)")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels([f"Seed {s}" for s in seeds])
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.6)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9, color="#d62728")
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9, color="#2ca02c")

# Bar chart: average comparison
axes[1].bar(["Untrained\n(Baseline)", "GRPO\n(Unsloth)"], [baseline_avg, trained_avg],
            color=["#d62728", "#2ca02c"], alpha=0.8, width=0.5)
axes[1].set_ylabel("Average Reward")
axes[1].set_title(f"Average Reward ({improvement:+.1f}% improvement)")
axes[1].grid(axis="y", linestyle="--", alpha=0.6)
axes[1].text(0, baseline_avg + 0.5, f"{baseline_avg:.1f}", ha="center", va="bottom", fontweight="bold")
axes[1].text(1, trained_avg + 0.5, f"{trained_avg:.1f}", ha="center", va="bottom", fontweight="bold")

# Plot 2: Training reward curve
history = pd.DataFrame(trainer.state.log_history)
if "rewards/reward_fn/mean" in history.columns:
    reward_steps = history.dropna(subset=["rewards/reward_fn/mean"])
    axes[2].plot(reward_steps["step"], reward_steps["rewards/reward_fn/mean"],
                 label="Mean Reward", color="#2ca02c", linewidth=2)
    axes[2].fill_between(reward_steps["step"],
                         reward_steps["rewards/reward_fn/mean"] - reward_steps["rewards/reward_fn/std"],
                         reward_steps["rewards/reward_fn/mean"] + reward_steps["rewards/reward_fn/std"],
                         alpha=0.2, color="#2ca02c")
    axes[2].axhline(y=baseline_avg, color="#d62728", linestyle="--", linewidth=1.5, label=f"Baseline ({baseline_avg:.1f})")
    axes[2].set_xlabel("Training Steps")
    axes[2].set_ylabel("Reward")
    axes[2].set_title("GRPO Training Curve (Unsloth)")
    axes[2].legend()
    axes[2].grid(True, linestyle="--", alpha=0.6)
else:
    axes[2].text(0.5, 0.5, "No reward history\n(run trainer.train() first)",
                 ha="center", va="center", transform=axes[2].transAxes, fontsize=12)
    axes[2].set_title("GRPO Training Curve")

plt.tight_layout()
plt.savefig("crisisinbox_grpo_unsloth_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Results saved to crisisinbox_grpo_unsloth_results.png")

## Evaluate Against Live Environment

Run the trained model in a closed loop against the actual CrisisInbox environment to get real server-side rewards.

In [ ]:
# Additional live eval on a fresh seed (not used in baseline comparison)
print("=== Extra Live Evaluation (seed=123) ===\n")
extra_result = evaluate_on_live_env(model, tokenizer, BASE_URL, seed=123, max_steps=25)
print(f"\nTotal reward: {extra_result['total_reward']:.1f}")
print(f"Actions taken: {len(extra_result['actions'])}")
print(f"Messages handled: {extra_result['final_status']['messages_handled']}")

In [ ]:
# Save the trained LoRA adapter
model.save_pretrained("crisisinbox-grpo-unsloth-lora")
tokenizer.save_pretrained("crisisinbox-grpo-unsloth-lora")
print("LoRA adapter saved to crisisinbox-grpo-unsloth-lora/")

In [ ]:
# Save merged model (LoRA weights merged into base model) for standalone deployment
# Saves as 16-bit for compatibility — use quantized_16bit for lower VRAM at inference
model.save_pretrained_merged(
    "crisisinbox-grpo-unsloth-merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved to crisisinbox-grpo-unsloth-merged/")

In [ ]:
# Optional: push to HuggingFace Hub
# model.push_to_hub_merged("eptan/crisisinbox-grpo-unsloth", tokenizer, save_method="merged_16bit")
# print("Pushed to HuggingFace Hub")